# AI–Quantum Portfolio — GitHub Code + Local Computer Data

Notebook này luôn lấy **mã nguồn hệ thống từ GitHub**, nhưng lấy **dữ liệu do người dùng tải từ máy cá nhân hoặc Google Drive**. Colab không thể truy cập trực tiếp ổ `C:` hoặc `D:` của Windows, vì vậy dữ liệu phải được đóng gói ZIP rồi upload/mount Drive.

File khuyến nghị trên máy hiện tại: `D:\NCKH 2026 - Thầy Dã\quantum_portfolio_data\colab_bundle\data_17_8_runtime.zip` (khoảng 18 MB). Không cần upload kho raw disclosure/PDF 54,6 GB để train/backtest mô hình chính.

In [ ]:
# CẤU HÌNH
from pathlib import Path

REPO_URL = 'https://github.com/23022006muki/AI-Quantum---Finance-Portfolio-Optimization.git'
SOURCE_COMMIT = 'a25429d33ca72721d1d4f8e4080abf207b89ecd2'
DATA_IMPORT_MODE = 'upload'  # 'upload' hoặc 'drive'
DRIVE_ZIP_PATH = '/content/drive/MyDrive/data_17_8_runtime.zip'
EXPECTED_DATA_ZIP_SHA256 = '4db33cf993977a9e55baf9e7c12ea52a4bc7adb70fe7fd8d29c4ff7188908f30'  # Đặt '' nếu dùng ZIP khác
RUN_TESTS = True
RUN_FULL_PIPELINE = True

REPO_DIR = Path('/content/AI-Quantum-Finance-Portfolio-Optimization')
PROJECT_DIR = REPO_DIR / 'quantum_portfolio_data'
TARGET_WORKSPACE = PROJECT_DIR / 'outputs' / 'Data 17_8'
print({'source_commit': SOURCE_COMMIT, 'data_mode': DATA_IMPORT_MODE, 'run_full': RUN_FULL_PIPELINE})

## 1. Clone mã nguồn đã commit trên GitHub

In [ ]:
import os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual == SOURCE_COMMIT
print('GitHub source verified:', actual)

## 2. Cài đầy đủ thư viện của hệ thống

In [ ]:
PINNED_PACKAGES = [
    'numpy==2.2.6', 'pandas==2.3.1', 'pyarrow==24.0.0',
    'scikit-learn==1.8.0', 'scipy==1.16.1', 'xgboost==3.3.0',
    'matplotlib==3.10.5', 'PyYAML==6.0.2', 'requests==2.32.5',
    'vnstock==4.0.5', 'finance-datareader==0.9.202', 'pip-system-certs==5.3',
    'streamlit==1.58.0', 'pypdf==6.1.3', 'pdfplumber==0.11.7',
    'Pillow==11.3.0', 'pytest==9.1.1'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *PINNED_PACKAGES], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Dependencies installed.')

## 3. Đưa dữ liệu từ máy cá nhân vào Colab

Với `DATA_IMPORT_MODE='upload'`, chạy cell và chọn `data_17_8_runtime.zip` trên máy. Với dữ liệu lớn hơn, upload ZIP lên Google Drive, đổi sang `'drive'` và chỉnh `DRIVE_ZIP_PATH`.

In [ ]:
if DATA_IMPORT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Hãy upload đúng một file ZIP dữ liệu.')
    DATA_ZIP = Path('/content') / zip_names[0]
elif DATA_IMPORT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ZIP = Path(DRIVE_ZIP_PATH)
else:
    raise ValueError("DATA_IMPORT_MODE phải là 'upload' hoặc 'drive'.")
if not DATA_ZIP.exists():
    raise FileNotFoundError(DATA_ZIP)
print('Selected local-data archive:', DATA_ZIP, f'({DATA_ZIP.stat().st_size / 1e6:.1f} MB)')

## 4. Kiểm tra hash, giải nén an toàn và nhận diện workspace
Notebook không giả định ZIP phải có đúng một cấp thư mục. Nó tìm `outputs/normalized/prices.parquet`, xác định gốc workspace và chép vào vị trí mà CLI sử dụng.

In [ ]:
import hashlib, json, zipfile

def sha256(path, block=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(block), b''):
            digest.update(chunk)
    return digest.hexdigest()

data_zip_hash = sha256(DATA_ZIP)
print('Local data ZIP SHA-256:', data_zip_hash)
if EXPECTED_DATA_ZIP_SHA256:
    assert data_zip_hash.lower() == EXPECTED_DATA_ZIP_SHA256.lower(), 'Data archive checksum mismatch'

EXTRACT_DIR = Path('/content/local_data_extract')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(DATA_ZIP) as zf:
    root = EXTRACT_DIR.resolve()
    for entry in zf.infolist():
        destination = (EXTRACT_DIR / entry.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError(f'Unsafe ZIP entry: {entry.filename}')
    bad_member = zf.testzip()
    if bad_member:
        raise ValueError(f'Corrupt ZIP member: {bad_member}')
    zf.extractall(EXTRACT_DIR)

price_candidates = [p for p in EXTRACT_DIR.rglob('prices.parquet') if p.parent.name == 'normalized' and p.parent.parent.name == 'outputs']
if len(price_candidates) != 1:
    raise ValueError(f'Expected one outputs/normalized/prices.parquet, found {len(price_candidates)}')
SOURCE_WORKSPACE = price_candidates[0].parent.parent.parent
if TARGET_WORKSPACE.exists():
    shutil.rmtree(TARGET_WORKSPACE)
shutil.copytree(SOURCE_WORKSPACE, TARGET_WORKSPACE)
print('Detected source workspace:', SOURCE_WORKSPACE)
print('Installed local data at:', TARGET_WORKSPACE)

## 5. Kiểm thử mã và audit dữ liệu local
Pipeline chỉ chạy nếu price panel local vượt qua exploratory gate. Các thiếu hụt enrichment vẫn được công khai, không được tự động điền giả.

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, '-m', 'src.cli', 'audit-data-17-8'], cwd=PROJECT_DIR, check=True)
audit_path = TARGET_WORKSPACE / 'outputs' / 'reports' / 'DATA_17_8_AUDIT.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
if not audit.get('exploratory_run_permitted'):
    raise RuntimeError('Local data failed the exploratory price-panel gate: ' + repr(audit.get('blockers')))
print('Exploratory run permitted:', audit['exploratory_run_permitted'])
print('Confirmatory status:', audit['status'])
print('Remaining blockers:', audit.get('blockers', []))

## 6. Mô tả bộ dữ liệu thực sự được đưa vào nghiên cứu

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display, Image

pd.set_option('display.max_columns', 100)
normalized = TARGET_WORKSPACE / 'outputs' / 'normalized'
prices = pd.read_parquet(normalized / 'prices.parquet')
benchmark = pd.read_parquet(normalized / 'benchmark.parquet') if (normalized / 'benchmark.parquet').exists() else pd.DataFrame()
actions = pd.read_parquet(normalized / 'corporate_actions.parquet') if (normalized / 'corporate_actions.parquet').exists() else pd.DataFrame()
full_master = pd.read_parquet(normalized / 'security_master_full.parquet') if (normalized / 'security_master_full.parquet').exists() else pd.read_parquet(normalized / 'security_master.parquet')
date_col = 'date' if 'date' in prices.columns else 'trading_date'
ticker_col = 'ticker' if 'ticker' in prices.columns else 'symbol'
dataset_summary = pd.DataFrame({
    'Chỉ tiêu': ['SHA-256 file ZIP local','Số quan sát','Số cổ phiếu runtime','Số mã security master','Ngày bắt đầu','Ngày kết thúc','Số phiên benchmark','Số corporate actions','Exploratory gate','Confirmatory audit'],
    'Kết quả': [data_zip_hash,len(prices),prices[ticker_col].nunique(),len(full_master),str(pd.to_datetime(prices[date_col]).min().date()),str(pd.to_datetime(prices[date_col]).max().date()),len(benchmark),len(actions),audit.get('exploratory_run_permitted'),audit.get('status')]
})
display(dataset_summary)
display(prices.head())
display(Markdown('Dữ liệu hiển thị ở đây được đọc từ ZIP vừa upload, không đọc từ `colab_bundle` trong GitHub. GitHub chỉ cung cấp source code trong notebook này.'))

## 7. Chạy toàn bộ hệ thống trên dữ liệu từ máy
Pipeline gồm feature engineering theo fold, XGBoost ranking, EWMA covariance đa biến, AUR, QUBO, Exact/SA/Penalty-QAOA/XY-QAOA, classical weighting, 33-fold walk-forward, ablation, sensitivity và bootstrap/Holm.

In [ ]:
experiments_root = TARGET_WORKSPACE / 'outputs' / 'experiments'
experiments_root.mkdir(parents=True, exist_ok=True)
before = {p.resolve() for p in experiments_root.iterdir() if p.is_dir()}
if RUN_FULL_PIPELINE:
    command = [sys.executable, '-m', 'src.cli', 'run-data-17-8', '--config', 'configs/data_17_8.yaml']
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    after = {p.resolve() for p in experiments_root.iterdir() if p.is_dir()}
    created = sorted(after - before, key=lambda p: p.stat().st_mtime)
    if not created:
        raise RuntimeError('Pipeline completed without a new experiment directory.')
    ACTIVE_EXPERIMENT = created[-1]
else:
    existing = list(experiments_root.iterdir())
    if not existing:
        raise RuntimeError('RUN_FULL_PIPELINE=False but local ZIP contains no experiment artifact.')
    ACTIVE_EXPERIMENT = max(existing, key=lambda p: p.stat().st_mtime)
print('Active experiment:', ACTIVE_EXPERIMENT)

## 8. Kết quả mô hình, solver và danh mục

In [ ]:
manifest = json.loads((ACTIVE_EXPERIMENT / 'manifest.json').read_text(encoding='utf-8'))
rankings = pd.read_csv(ACTIVE_EXPERIMENT / 'rankings.csv')
tests = pd.read_csv(ACTIVE_EXPERIMENT / 'statistical_tests.csv')
comparisons = pd.read_csv(ACTIVE_EXPERIMENT / 'comparisons.csv')
metrics = pd.read_csv(ACTIVE_EXPERIMENT / 'strategy_metrics_summary.csv')

rank_by_fold = rankings.groupby('fold')[['xgboost_rank_ic','ewma_rank_ic']].first()
signal_summary = pd.DataFrame({
    'Model':['XGBoost','EWMA'],
    'Mean Rank IC':[rank_by_fold.xgboost_rank_ic.mean(),rank_by_fold.ewma_rank_ic.mean()],
    'Median Rank IC':[rank_by_fold.xgboost_rank_ic.median(),rank_by_fold.ewma_rank_ic.median()]
})
display(Markdown(f"**Experiment `{manifest['experiment_id']}`:** {manifest['folds_completed']}/{manifest['folds_requested']} folds, OOS {manifest['actual_oos_start']}–{manifest['actual_oos_end']}."))
display(signal_summary.style.format({'Mean Rank IC':'{:.4f}','Median Rank IC':'{:.4f}'}))

solver_labels={'exact':'Exact','simulated_annealing':'Simulated Annealing','penalty_stochastic_baseline':'Penalty stochastic baseline','penalty_qaoa_ideal_statevector':'Penalty-QAOA','xy_qaoa_dicke_ideal_statevector':'XY-QAOA + Dicke'}
solver_table=comparisons.copy()
solver_table['Solver']=solver_table.method.map(solver_labels).fillna(solver_table.method)
display(solver_table[['Solver','runs','feasibility_rate','optimality_gap_mean','runtime_seconds']].style.format({'feasibility_rate':'{:.2%}','optimality_gap_mean':'{:.2%}','runtime_seconds':'{:.4f}'}))

preferred=['full_pipeline_xy_qaoa','benchmark_vnallsharetri','liquidity_topk_exact','minimum_variance','equal_weight_universe','ewma_topk_exact','adaptive_exact','xgboost_topk_exact','xgboost_penalty_qaoa']
portfolio_table=metrics[metrics.strategy.isin(preferred)].copy()
portfolio_table['order']=portfolio_table.strategy.map({x:i for i,x in enumerate(preferred)})
portfolio_table=portfolio_table.sort_values('order')[['strategy','cumulative_return','annualized_return','annualized_volatility','sharpe','max_drawdown','turnover','total_cost']]
display(portfolio_table.style.format({'cumulative_return':'{:.2%}','annualized_return':'{:.2%}','annualized_volatility':'{:.2%}','sharpe':'{:.4f}','max_drawdown':'{:.2%}','turnover':'{:.2f}','total_cost':'{:.2%}'}))
for name in ['equity_curve.png','drawdown.png','risk_return.png']:
    figure=ACTIVE_EXPERIMENT/'figures'/name
    if figure.exists(): display(Image(filename=str(figure)))

latest=pd.read_csv(ACTIVE_EXPERIMENT/'latest_selected_portfolio.csv')
latest_summary=json.loads((ACTIVE_EXPERIMENT/'latest_portfolio_summary.json').read_text(encoding='utf-8'))
basket=latest[['ticker','company_name','target_weight','adv_participation']].copy()
cash=latest_summary.get('executed_cash_weight',1-basket.target_weight.sum())
basket=pd.concat([basket,pd.DataFrame([{'ticker':'CASH','company_name':'Tiền mặt','target_weight':cash,'adv_participation':np.nan}])],ignore_index=True)
display(Markdown(f"**Rổ cuối — ngày quyết định {latest.decision_time.iloc[0]}**"))
display(basket.style.format({'target_weight':'{:.2%}','adv_participation':'{:.4%}'},na_rep='—'))

## 9. Kết luận H1–H6 theo kiểm định của lần chạy local

In [ ]:
def row(name): return tests.loc[tests.test.eq(name)].iloc[0]
h1=row('xgboost_rank_ic_vs_ewma_rank_ic')
h2r=row('adaptive_universe_forward_return_vs_fixed_topm')
h2d=row('adaptive_universe_diversification_vs_fixed_topm')
h3=row('xy_feasibility_vs_penalty_qaoa')
h4=row('xy_optimality_gap_vs_penalty_qaoa')
h5=tests.loc[tests.hypothesis.eq('H5')]
text=f'''
### H1 — {'Được hỗ trợ' if h1.p_value_holm < 0.05 else 'Không được hỗ trợ'}
Chênh lệch Rank IC XGBoost–EWMA là {h1.mean_difference:.4f}, CI [{h1.ci_low:.4f}, {h1.ci_high:.4f}], p-Holm={h1.p_value_holm:.3f}.

### H2 — {'Được hỗ trợ đầy đủ' if h2r.p_value_holm < 0.05 and h2d.p_value_holm < 0.05 else 'Được hỗ trợ một phần' if h2r.p_value_holm < 0.05 or h2d.p_value_holm < 0.05 else 'Không được hỗ trợ'}
AUR có kiểm định forward-return p-Holm={h2r.p_value_holm:.3f} và kiểm định diversification p-Holm={h2d.p_value_holm:.3f}; hai cấu phần được kết luận riêng.

### H3 — {'Được hỗ trợ trong simulator lý tưởng' if h3.p_value_holm < 0.05 else 'Không được hỗ trợ'}
Chênh lệch feasibility XY-QAOA so với Penalty-QAOA là {h3.mean_difference:.4f}, p-Holm={h3.p_value_holm:.3f}.

### H4 — {'Được hỗ trợ có điều kiện so với Penalty-QAOA' if h4.p_value_holm < 0.05 else 'Không được hỗ trợ'}
Mức cải thiện optimality gap là {h4.mean_difference:.4f}, p-Holm={h4.p_value_holm:.3f}; kết luận không hàm ý vượt Exact/SA.

### H5 — {'Được hỗ trợ trong ít nhất một so sánh' if (h5.p_value_holm < 0.05).any() else 'Không được hỗ trợ'}
Số so sánh tài chính có ý nghĩa sau Holm là {(h5.p_value_holm < 0.05).sum()}/{len(h5)}. Lợi nhuận dương nếu có không tự động được xem là alpha thống kê.

### H6 — Phân tích độ nhạy đã hoàn thành
Kết quả chỉ được diễn giải trong lưới depth, shots, seed, cardinality, noise và chi phí đã chạy; H6 phù hợp hơn với câu hỏi robustness.
'''
display(Markdown(text))

## 10. Tải toàn bộ output của lần chạy về máy

In [ ]:
result_base=Path('/content')/f"local_data_ai_quantum_{manifest['experiment_id']}"
result_zip=shutil.make_archive(str(result_base),'zip',root_dir=ACTIVE_EXPERIMENT)
print('Created:',result_zip,f'({Path(result_zip).stat().st_size/1e6:.1f} MB)')
from google.colab import files
display(Markdown(f"Chạy `files.download('{result_zip}')` nếu muốn tải ngay toàn bộ output."))